# Paper Sleeping Beauty (Ke et al. 2015) — Beauty coefficient B + awakening time T

For every cited publication, its yearly citation histogram `C[age]` (`age = citing_year −
cited_year ≥ 0`) and the Beauty coefficient `SB_B` / awakening time `SB_T`, with the kernel of
`OpenAlex/notebook/paper_sb.ipynb` copied verbatim. Primary key: `paper_id`.

## Input
```
Dimensions/cache/paper_csr.npz   # in_ptr, in_idx (citers per cited), year, uni_mag
```

## Metric
`t_m = argmax_t C[t]`, `c_m = C[t_m]`, `c_0 = C[0]`; `t_m = 0` → `B = 0, T = 0`; otherwise
$$B=\sum_{t=0}^{t_m}\frac{\frac{c_m-c_0}{t_m}\,t + c_0 - C[t]}{\max(C[t],1)},\qquad
T=\arg\max_{0\le t\le t_m}\frac{|(c_m-c_0)\,t + (c_0-C[t])\,t_m|}{\sqrt{(c_m-c_0)^2+t_m^2}}.$$

## Output
`Dimensions/output/paper_sb.parquet` — `paper_id, SB_B, SB_T, n_cite` (uncited publications omitted).

In [1]:
import os, sys, gc, glob, time
import numpy as np, pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/Dimensions')
import dim_common as dim
ROOT = dim.BASE; OUT = dim.OUT
print('dump:', dim.ROOT)
OUT_FP = f'{OUT}/paper_sb.parquet'
# Everything below is fed by notebook/references_w_year.ipynb: it writes the per-publication
# map (year + source), the scalar and author parts, and the edge table with both years and both
# source ids. Build it once before running this notebook.
assert dim.have_consolidated(), (
    'run notebook/references_w_year.ipynb first -- it builds the map, the scalar parts and the edge table')
dim.summary()

dump: /project/jevans/dimensions/dimensions/dimensions_june_2025
dump     : /project/jevans/dimensions/dimensions/dimensions_june_2025
cache    : /project/jevans/Dawoon/Science of Science/Dimensions/cache
output   : /project/jevans/Dawoon/Science of Science/Dimensions/output
  consolidated edge table: present
  map            2.80 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/pub_year_source_map.npz
  graph         19.00 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_graph.npz
  csr           21.49 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
  journal        1.24 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_journal.parquet
  fos            0.77 GB  /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_fos.parquet
  pat2pub      not built  /project/jevans/Dawoon/Science of Science/Dimensions/cache/patent2pub_edges.parquet
  scalars       4219 parts  /project/jevans/Dawoon/Science o

In [2]:
from numba import njit, prange

@njit
def _sb_one(C):
    """C[age] = #citations received at that age (0..len-1). Returns (B, T). Mirrors MAG-SB.ipynb."""
    m = len(C)
    t_m = 0; cmax = C[0]
    for t in range(m):
        if C[t] > cmax:
            cmax = C[t]; t_m = t
    if t_m == 0:
        return 0.0, 0            # peaks at publication -> not a sleeping beauty
    c_m = C[t_m]; c_0 = C[0]
    B = 0.0
    for t in range(t_m + 1):
        den = C[t] if C[t] != 0.0 else 1.0
        B += ((c_m - c_0) / t_m * t + c_0 - C[t]) / den
    norm = np.sqrt((c_m - c_0) ** 2 + t_m * t_m)
    T = 0; dmax = -1.0
    for t in range(t_m + 1):
        d = abs((c_m - c_0) * t + (c_0 - C[t]) * t_m) / norm
        if d > dmax:
            dmax = d; T = t
    return B, T

@njit(parallel=True)
def sb_from_age_csr(ptr, ages, cnts, SBB, SBT, NC):
    """For each work w, its citation ages are ages[ptr[w]:ptr[w+1]] with counts cnts[...]."""
    n = len(ptr) - 1
    for w in prange(n):
        a0 = ptr[w]; a1 = ptr[w + 1]
        if a1 == a0:
            continue
        mx = -1; tot = 0
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                tot += cnts[j]
                if ag > mx:
                    mx = ag
        if mx < 0:
            continue
        C = np.zeros(mx + 1, np.float64)
        for j in range(a0, a1):
            ag = ages[j]
            if ag >= 0:
                C[ag] += cnts[j]
        B, T = _sb_one(C)
        SBB[w] = B; SBT[w] = T; NC[w] = tot

## 1. Load cached CSR (citers per cited) and run the SB kernel

In [3]:
%%time
out_ptr, out_idx, in_ptr, in_idx, year, uni_mag = dim.load_csr()
n = len(uni_mag); print(f'publications: {n:,}, citation edges: {len(in_idx):,}')

CSR cache present: /project/jevans/Dawoon/Science of Science/Dimensions/cache/paper_csr.npz
publications: 155,441,856, citation edges: 2,141,693,663


In [4]:
%%time
from numba import njit, prange
@njit(parallel=True)
def sb_paper(in_ptr, in_idx, year, SBB, SBT, NC):
    n = len(in_ptr) - 1
    for w in prange(n):
        a0 = in_ptr[w]; a1 = in_ptr[w + 1]
        if a1 == a0:
            continue
        yW = year[w]; mx = -1; tot = 0
        for j in range(a0, a1):
            ag = year[in_idx[j]] - yW
            if ag >= 0:
                tot += 1
                if ag > mx: mx = ag
        if mx < 0:
            continue
        Cc = np.zeros(mx + 1, np.float64)
        for j in range(a0, a1):
            ag = year[in_idx[j]] - yW
            if ag >= 0: Cc[ag] += 1.0
        B, T = _sb_one(Cc)
        SBB[w] = B; SBT[w] = T; NC[w] = tot

SBB = np.full(n, np.nan, np.float64); SBT = np.full(n, -1, np.int32); NC = np.zeros(n, np.int64)
t = time.time(); sb_paper(in_ptr, in_idx, year, SBB, SBT, NC)   # first call JIT-compiles then runs
print(f'SB computed in {time.time()-t:.0f}s (incl. JIT compile)')

SB computed in 14s (incl. JIT compile)


## 2. Assemble + save

In [5]:
mask = NC > 0
out = pd.DataFrame({'paper_id': dim.code_to_id(uni_mag[mask]),
                    'SB_B': SBB[mask].astype(np.float32), 'SB_T': SBT[mask], 'n_cite': NC[mask]})
out.to_parquet(OUT_FP, index=False)
print(f'WROTE {OUT_FP}  ({len(out):,} rows)')
print('SB_B summary:'); print(out['SB_B'].describe().round(3).to_string())
print('top sleeping beauties:'); display(out.nlargest(10, 'SB_B'))

WROTE /project/jevans/Dawoon/Science of Science/Dimensions/output/paper_sb.parquet  (83,849,678 rows)
SB_B summary:
count    8.384968e+07
mean     5.124000e+00
std      2.748300e+01
min     -1.331100e+01
25%      0.000000e+00
50%      1.000000e+00
75%      4.000000e+00
max      3.801115e+04
top sleeping beauties:


,paper_id,SB_B,SB_T,n_cite
55577248,pub.1099370191,38011.152344,242,3142
51160478,pub.1087190994,35911.648438,206,7814
62669709,pub.1120238898,33379.792969,118,1986
35533299,pub.1055805656,31320.673828,50,28365
18061541,pub.1028083705,27283.025391,196,5999
67575273,pub.1131651734,22497.791016,82,1564
51595520,pub.1090110805,21552.513672,96,6185
32445568,pub.1050450827,20934.261719,101,9326
77073540,pub.1154284703,19933.794922,55,2868
6454485,pub.1010033334,19237.435547,232,3523
